# Static Word Embeddings

This notebook explores pretrained static word embeddings using
the adapted conversational corpus introduced in Notebook 1.

Unlike Bag-of-Words and TF-IDF, static embeddings represent words
as dense numerical vectors learned from patterns of word co-occurrence.

The analysis examines vocabulary coverage, lexical similarity,
and the treatment of novel linguistic expressions, particularly
*frankenstein* and *frankensteining*.

The objective is to demonstrate the application and limitations
of pretrained static embeddings, rather than train a model on
the small project corpus.

## 1. Pretrained model and vocabulary coverage

We use spaCy's `en_core_web_md` English pipeline, which provides 300-dimensional static word vectors learned from external text data. No additional training is performed on our conversational corpus.

Our first objective is to determine whether pretrained vectors are available for the two target expressions:

- *frankenstein*: an existing lexical form used innovatively as a verb in the corpus.
- *frankensteining*: a derived *-ing* form used nominally in subsequent exchanges.

We begin by checking vector availability and vector norms. A non-zero vector norm indicates that a token has a non-zero numerical representation, while `has_vector` reports whether a vector is available.

We then examine how spaCy tokenizes the expressions. Tokenization and vector availability are distinct properties: a word may be recognized as a single token without having a pretrained vector.

In [1]:
# Check whether there are vectors for target terms.

import spacy

nlp = spacy.load("en_core_web_md")

target_words = ["frankenstein", "frankensteining"]

for word in target_words:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Vector norm:", round(token.vector_norm, 4)
    )

frankenstein | Has vector: True | Vector norm: 6.3077
frankensteining | Has vector: False | Vector norm: 0.0


In [2]:
for word in target_words:
    doc = nlp(word)
    print(word, "→", [token.text for token in doc])

frankenstein → ['frankenstein']
frankensteining → ['frankensteining']


1.1. Findings

Both target expressions were tokenized as single tokens by spaCy. However, their pretrained vector coverage differed:

* frankenstein: A pretrained vector was available, with a vector norm of 6.3077.
* frankensteining: No pretrained vector was available (has_vector = False; vector norm = 0.0).

This distinction illustrates that successful tokenization does not guarantee the availability of a pretrained lexical representation.

Although frankensteining is morphologically related to frankenstein, the static embedding model does not automatically derive a representation for the novel form from its constituent elements.

Consequently, the following lexical similarity analysis focuses on frankenstein, the only target expression with an available pretrained vector.

2. Lexical similarity

Having established that frankenstein has an available pretrained vector, we examine its similarity to six selected reference words.

The reference words represent two sets of semantic associations:

* Literary associations: monster, horror and creature, reflecting the established cultural associations of Frankenstein.
* Corpus-related associations: reconstruction, combination and modification, reflecting aspects of the innovative meaning developed in the conversational corpus.

We calculate cosine similarity between the static vector for frankenstein and each reference word.

Cosine similarity measures the directional alignment of two vectors. Higher values indicate greater alignment in the embedding space, but do not independently establish a particular semantic relationship.

The comparison is exploratory: it examines whether the pretrained representation is more closely aligned with either group of selected words, without assuming that the model has acquired the meaning developed in our conversations.

In [3]:
reference_word = nlp("frankenstein")[0]

comparison_words = [
    "monster",
    "horror",
    "creature",
    "reconstruction",
    "combination",
    "modification"
]

for word in comparison_words:
    token = nlp(word)[0]

    print(
        word,
        "| Similarity:",
        round(reference_word.similarity(token), 4)
    )

monster | Similarity: 0.137
horror | Similarity: 0.0981
creature | Similarity: 0.0559
reconstruction | Similarity: 0.0992
combination | Similarity: 0.0555
modification | Similarity: 0.0571


### 2.1. Findings

The cosine similarity comparisons between *frankenstein* and the six selected reference words produced relatively low values:

| Reference word | Cosine similarity |
|---|---:|
| monster | 0.1370 |
| horror | 0.0981 |
| creature | 0.0559 |
| reconstruction | 0.0992 |
| combination | 0.0555 |
| modification | 0.0571 |

The highest similarity was observed with *monster* (0.1370), followed by *reconstruction* (0.0992) and *horror* (0.0981).

The results do not show a clear separation between the literary and corpus-related reference words. Although *monster* has the highest similarity, the remaining values are relatively close, and neither group consistently exhibits greater alignment with *frankenstein*.

These comparisons therefore do not establish a strong semantic association with either the established literary meaning or the innovative meaning developed in the conversational corpus.

The analysis is limited to six manually selected reference words and the pretrained static representation of *frankenstein*. It does not provide evidence about how the expression is interpreted in individual conversational contexts.

## 3. Capitalization

The preceding analysis used the lowercase form *frankenstein*. However, the expression originates from the proper name *Frankenstein*, which is conventionally capitalized.

We therefore examine whether capitalization affects the pretrained static representation.

Using the same spaCy model, we compare *frankenstein* and *Frankenstein* in terms of vector availability, vector norm and cosine similarity with *monster*.

If both forms produce identical results, this would suggest that capitalization does not distinguish their representations in these comparisons.

This check does not determine whether the model distinguishes the literary proper name from the innovative verbal use in context, since static word vectors do not change according to the grammatical or discourse environment in which a word occurs.

In [4]:

for word in ["frankenstein", "Frankenstein"]:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Vector norm:", round(token.vector_norm, 4),
        "| Similarity to monster:",
        round(token.similarity(nlp("monster")[0]), 4)
    )

frankenstein | Has vector: True | Vector norm: 6.3077 | Similarity to monster: 0.137
Frankenstein | Has vector: True | Vector norm: 6.3077 | Similarity to monster: 0.137


### 3.1. Findings

Both *frankenstein* and *Frankenstein* have pretrained vectors available in spaCy's `en_core_web_md` model.

The two forms produced identical results for the measurements examined:

- **Vector norm:** 6.3077
- **Cosine similarity with *monster*:** 0.1370

These findings indicate that capitalization does not affect the measured vector norms or similarity scores in this comparison.

However, identical vector norms and similarity scores do not necessarily imply identical numerical vectors. Establishing that would require a direct comparison of their vector components.

The results therefore provide no evidence of a capitalization-related distinction in the measurements examined, but they do not establish that the two forms have identical representations.

## 4. Nearest neighbours

The preceding comparisons examined the similarity between *frankenstein* and a small set of manually selected reference words.

We now reverse the approach: rather than selecting comparison words in advance, we retrieve vocabulary entries whose pretrained vectors are closest to the vector for *frankenstein*.

This nearest-neighbour search uses cosine similarity to identify closely aligned representations in spaCy's static embedding space.

The objective is to investigate which lexical entries occupy the same or nearby directions in the vector space and whether these relationships provide interpretable information about the representation of *frankenstein*.

As with the preceding comparisons, vector proximity should not be interpreted as direct evidence of semantic equivalence.

In [5]:
import numpy as np

# Get the vector for our target word
target_vector = nlp("frankenstein")[0].vector

# Find the five nearest vectors
keys, _, scores = nlp.vocab.vectors.most_similar(
    np.asarray([target_vector], dtype="float32"),
    n=5
)

# Display the results
for key, score in zip(keys[0], scores[0]):
    print(nlp.vocab.strings[int(key)], round(float(score), 4))

Terrance 1.0
eastwood 1.0
Roosevelt 1.0
julius 0.7495
HEATH 0.7495


### 4.1. Findings

The nearest-neighbour search produced an unexpected result: several vocabulary entries with no obvious semantic relationship to *frankenstein* were retrieved with a cosine similarity of **1.0**.

These included *Terrance*, *eastwood* and *Roosevelt*, despite their apparent lack of connection to either the established literary associations of *Frankenstein* or the innovative meaning developed in our corpus.

This contrasts sharply with the manually selected reference words examined in Section 2, where the highest similarity was only **0.1370** (*monster*).

A cosine similarity of 1.0 indicates that two nonzero vectors have the same direction, within numerical precision. However, this does not establish semantic equivalence between the corresponding words.

The unexpected results raise a further question: **do these vocabulary entries actually share identical numerical vectors, or do they merely have closely aligned representations?**

To investigate this, we examine their vector values and assignments in spaCy's underlying vector table.

## 5. Investigating identical vectors

The nearest-neighbour search identified several apparently unrelated vocabulary entries with a cosine similarity of 1.0 to *frankenstein*.

To investigate this result, we examine whether the similarity reflects identical numerical representations and how these representations are stored in spaCy's vector table.

We select four entries for closer inspection: *frankenstein*, *Terrance*, *eastwood* and *Roosevelt*.

The investigation proceeds in two stages:

1. **Vector comparison:** We compare the numerical components of their 300-dimensional vectors to determine whether the representations are identical, rather than merely aligned in the embedding space.
2. **Vector-table inspection:** We examine the vector-table row associated with each vocabulary entry to determine whether identical representations correspond to the same stored row or occur at different locations.

These checks distinguish two properties that cosine similarity alone cannot establish: numerical vector equality and shared vector-table assignments.

The objective is to characterize the unexpected nearest-neighbour results before considering possible explanations for their occurrence.

In [6]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Same vector:",
        np.array_equal(target_vector, token.vector)
    )

frankenstein | Has vector: True | Same vector: True
Terrance | Has vector: True | Same vector: True
eastwood | Has vector: True | Same vector: True
Roosevelt | Has vector: True | Same vector: True


In [7]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    lexeme = nlp.vocab[word]

    print(
        word,
        "| Vector row:",
        lexeme.rank
    )

frankenstein | Vector row: 14630
Terrance | Vector row: 14630
eastwood | Vector row: 14629
Roosevelt | Vector row: 14628


In [8]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    lexeme = nlp.vocab[word]
    row = nlp.vocab.vectors.key2row.get(lexeme.orth)

    print(word, "| Actual vector row:", row)

frankenstein | Actual vector row: 14630
Terrance | Actual vector row: 14630
eastwood | Actual vector row: 14629
Roosevelt | Actual vector row: 14628


### 5.1. Findings

The investigation confirmed that *frankenstein*, *Terrance*, *eastwood* and *Roosevelt* have identical numerical vector representations in the pretrained spaCy model.

This explains their cosine similarity of 1.0: the similarity reflects numerical identity rather than merely closely aligned vector directions.

The vector-table inspection revealed an additional distinction:

- **frankenstein** and **Terrance** were assigned to the same vector-table row (14630).
- **eastwood** and **Roosevelt** had different row assignments, despite sharing the same numerical vector values.

The two methods used to inspect row assignments, `lexeme.rank` and `nlp.vocab.vectors.key2row`, returned matching row numbers for the examined entries.

These findings demonstrate that identical numerical vectors do not necessarily correspond to a shared vector-table row.

The results are consistent with vocabulary compression or vector-table optimization in the pretrained model, although the inspection does not establish the precise mechanism responsible for these assignments.

Most importantly, the numerical identity of these representations should not be interpreted as evidence of semantic equivalence between the corresponding words.

The unexpected nearest-neighbour results therefore reveal a limitation of interpreting static embedding similarity without examining the underlying vector representations.

## 6. Findings and limitations

The investigation of spaCy's pretrained static embeddings revealed several characteristics relevant to the representation of creative lexical forms.

**Vocabulary coverage.** Both *frankenstein* and *frankensteining* were recognized as single tokens, but only *frankenstein* had an available pretrained vector. The derived form *frankensteining* could not be examined through an individual pretrained lexical vector in this model.

**Lexical similarity.** Comparisons between *frankenstein* and six manually selected reference words produced relatively low cosine similarities. The results did not establish a clear alignment with either the established literary associations of *Frankenstein* or the innovative meaning developed in the conversational corpus.

**Capitalization.** The lowercase and capitalized forms produced identical vector norms and similarity scores with *monster*. These measurements provided no evidence of a capitalization-related distinction, although the complete vectors were not directly compared.

**Nearest neighbours and vector identity.** The nearest-neighbour search retrieved apparently unrelated vocabulary entries with cosine similarities of 1.0. Further inspection confirmed identical numerical vectors for *frankenstein*, *Terrance*, *eastwood* and *Roosevelt*. Some entries shared a vector-table row, while others had different row assignments despite identical vector values.

Together, these findings illustrate several limitations of interpreting pretrained static embeddings as direct representations of lexical meaning.

In particular, the availability of a vector does not guarantee that its nearest neighbours will provide interpretable semantic relationships. Likewise, identical numerical representations do not establish that the corresponding lexical items have equivalent meanings.

A further limitation concerns **contextual variation**. Static embeddings assign a fixed representation to a lexical entry, regardless of its grammatical function, semantic reference or discourse environment. They therefore cannot represent the changing contextual uses of *Frankenstein* and *frankensteining* observed in our conversational corpus.

The next notebook addresses this limitation by examining contextual embeddings produced by DistilBERT. Unlike the static representation examined here, these embeddings vary according to the linguistic context in which an expression occurs.

This allows us to investigate representational differences between the original verbal innovation, subsequent nominal uses and the assistant-produced creative extension, while remaining cautious about what vector similarity can establish concerning linguistic meaning.